In [7]:
# MICrONS Visual Decoding
# This notebook contains functions in order to easily extract neuron data from
# MICrONS dataset. Along with LDA analysis on neurons.

!pip install -q dandi remfile pynwb h5py scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import requests
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

# ============================================================================
# CONNECT TO DATA
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    # Find the nwb file
    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    # Get the file URL and open it
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    print("Connected to dataset")
    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    rs = fluorescence.roi_response_series["RoiResponseSeries3"]

    print("Total neurons:", rs.data.shape[1])
    print("Total timepoints:", rs.data.shape[0])

    return rs


def get_stimulus_info(nwb):
    """Get information about what stimuli were shown."""
    clip_intervals = nwb.intervals['Clip']

    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])

    print("Total stimulus presentations:", len(clip_starts))

    # Count how many of each type
    unique_types = np.unique(clip_types)
    for stim_type in unique_types:
        count = 0
        for ct in clip_types:
            if ct == stim_type:
                count = count + 1
        print("  ", stim_type, ":", count, "presentations")

    return clip_starts, clip_stops, clip_types


# ============================================================================
# EXTRACT DATA FOR DECODING
# ============================================================================

def get_data_for_decoding(rs, timestamps, clip_starts, clip_stops, clip_types,
                         neuron_list, target_conditions, window_size=None):
    """
    Get neural data organized for decoding

    Parameters:
    - rs: neural recording data
    - timestamps: time values for each data point
    - clip_starts, clip_stops: when each stimulus started/stopped
    - clip_types: what type each stimulus was
    - neuron_list: list of which neurons to use (e.g., [3] for neuron 4, or [0,1,2] for first 3)
    - target_conditions: which stimulus types to include
    - window_size: how many seconds of data to use per trial (if None, uses exact clip duration)

    Returns:
    - X: neural data matrix (trials x neurons)
    - y: labels for each trial (0, 1, 2...)
    - condition_names: what each label number means
    """


    # Make a mapping from condition name to number
    condition_names = {}
    label_number = 0
    for cond in target_conditions:
        condition_names[cond] = label_number
        label_number = label_number + 1

    # Store data here
    all_trials = []
    all_labels = []

    # Go through each stimulus presentation
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        # Skip if not in our target conditions
        if stim_type not in target_conditions:
            continue

        # Find the time window
        start_time = clip_starts[i]
        if window_size is None:
            # Use exact clip duration
            stop_time = clip_stops[i]
        else:
            # Use fixed window
            stop_time = start_time + window_size

        # Convert times to indices
        start_idx = 0
        for t in range(len(timestamps)):
            if timestamps[t] >= start_time:
                start_idx = t
                break

        stop_idx = start_idx
        for t in range(start_idx, len(timestamps)):
            if timestamps[t] >= stop_time:
                stop_idx = t
                break

        # Get neural data for this window
        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

        # Average across time to get one number per neuron
        neural_average = []
        for neuron_idx in range(neural_chunk.shape[1]):
            neuron_data = neural_chunk[:, neuron_idx]
            avg = np.mean(neuron_data)
            neural_average.append(avg)

        # Store this trial
        all_trials.append(neural_average)
        all_labels.append(condition_names[stim_type])

    # Convert to arrays
    X = np.array(all_trials)
    y = np.array(all_labels)

    print("Data shape:", X.shape)
    print("Number of trials:", len(y))

    return X, y, condition_names

# ============================================================================
# TRAIN AND TEST DECODER
# ============================================================================

def train_decoder(X, y, condition_names):
    """
    Train a decoder and test it

    Parameters:
    - X: neural data (trials x neurons)
    - y: labels (trial numbers)
    - condition_names: dict mapping names to numbers

    Returns:
    - accuracy: how well the decoder did (0 to 1)
    - y_test: true labels for test trials
    - y_pred: predicted labels for test trials
    """

    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Training trials:", len(y_train))
    print("Test trials:", len(y_test))

    # Train the LDA model
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train, y_train)

    # Test the model
    y_pred = lda.predict(X_test)

    # Calculate accuracy
    correct = 0
    for i in range(len(y_test)):
        if y_test[i] == y_pred[i]:
            correct = correct + 1

    accuracy = correct / len(y_test)

    print("Test accuracy:", accuracy)

    return accuracy, y_test, y_pred, condition_names


def print_results(accuracy, y_test, y_pred, condition_names):
    """Print the results"""

    # Reverse the condition names dict
    number_to_name = {}
    for name, number in condition_names.items():
        number_to_name[number] = name

    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)

    n_conditions = len(condition_names)
    chance = 100.0 / n_conditions

    print("Overall accuracy:", round(accuracy * 100, 1), "%")

    # Per-condition results
    print("\nPer-condition accuracy:")
    for label_num in sorted(number_to_name.keys()):
        cond_name = number_to_name[label_num]

        # Count trials for this condition
        total = 0
        correct = 0
        for i in range(len(y_test)):
            if y_test[i] == label_num:
                total = total + 1
                if y_pred[i] == label_num:
                    correct = correct + 1

        if total > 0:
            cond_acc = correct / total
            print("  ", cond_name, ":", round(cond_acc * 100, 1), "%",
                  "(", correct, "out of", total, ")")

In [9]:
# ============================================================================
# ANAKYSIS
# ============================================================================

nwb = connect_to_data()                           # Load the NWB data file (standard neuroscience data format)
rs = get_neural_data(nwb)                         # Extract neural recordings (e.g., firing rates or calcium signals)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)  # Load video clip timing and type info

# Get a manageable number of timestamps (first 100,000 samples)
timestamps = np.array(rs.timestamps[:100000])

# Choose which types of stimuli to include in decoding
target_conditions = ['Cinematic', 'Rendered', 'sports1m']

# ============================================================================
# TEST 1
# ============================================================================
print("\n" + "="*60)
print("TEST 1: How many neurons do we need")
print("="*60)

neuron_counts = [10, 50, 100, 200, 500, 1000]

for n_neurons in neuron_counts:
    print("\nTesting with", n_neurons, "neurons")

    # Make a list of neuron indices (just the first N neurons)
    neuron_list = []
    for i in range(n_neurons):
        neuron_list.append(i)

    # Extract features (X) and labels (y) for decoding
    # Each trial corresponds to one video clip presentation
    X, y, cond_map = get_data_for_decoding(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        neuron_list=neuron_list,
        target_conditions=target_conditions,
        window_size=None  # use the full clip duratoin
    )

    # Train
    acc, y_test, y_pred, cond_map = train_decoder(X, y, cond_map)

    print("Result:", round(acc * 100, 1), "%")


# ============================================================================
# TEST 2
# ============================================================================
print("\n" + "="*60)
print("TEST 2: How long should time windows be")
print("="*60)

# Test different fixed time window sizes for averaging neural activity
window_sizes = [2.0, 5.0, 10.0]

# Use a fixed set of 100 neurons for consistency
neuron_list = []
for i in range(100):
    neuron_list.append(i)

for window_size in window_sizes:
    print("\nTesting with", window_size, "second window")

    # Extract neural data for the chosen window size
    X, y, cond_map = get_data_for_decoding(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        neuron_list=neuron_list,
        target_conditions=target_conditions,
        window_size=window_size
    )

    # Train decoder and print accuracy
    acc, y_test, y_pred, cond_map = train_decoder(X, y, cond_map)
    print("Result:", round(acc * 100, 1), "%")

# ============================================================================
# TEST 3
# ============================================================================
print("\n" + "="*60)
print("TEST 3: Neuron 4 vs Population")
print("="*60)

# Single neuron - using exact clip duration
print("\nSingle neuron (neuron 4)")
X_single, y_single, cond_map = get_data_for_decoding(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list=[3],  # Neuron 4
    target_conditions=target_conditions,
    window_size=None
)
acc_single, y_test, y_pred, cond_map = train_decoder(X_single, y_single, cond_map)
print_results(acc_single, y_test, y_pred, cond_map)

# All neurons - using exact clip duration
print("\nAll neurons (population)")
all_neurons = []
for i in range(rs.data.shape[1]):
    all_neurons.append(i)

X_pop, y_pop, cond_map = get_data_for_decoding(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list=all_neurons,
    target_conditions=target_conditions,
    window_size=None
)
acc_pop, y_test, y_pred, cond_map = train_decoder(X_pop, y_pop, cond_map)
print_results(acc_pop, y_test, y_pred, cond_map)


# ============================================================================
# TEST 4
# ============================================================================
print("\n" + "="*60)
print("TEST 4: Max vs Mean")
print("="*60)

def get_data_with_max(rs, timestamps, clip_starts, clip_stops, clip_types,
                      neuron_list, target_conditions, window_size=None):
    """Same as before but use max instead of mean."""

    condition_names = {}
    label_number = 0
    for cond in target_conditions:
        condition_names[cond] = label_number
        label_number = label_number + 1

    all_trials = []
    all_labels = []

    # loop through each stimulus
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        if window_size is None:
            # Use exact clip duration
            stop_time = clip_stops[i]
        else:
            # Use fixed window
            stop_time = start_time + window_size

        # Convert times to indicies in the neural data
        start_idx = 0
        for t in range(len(timestamps)):
            if timestamps[t] >= start_time:
                start_idx = t
                break

        stop_idx = start_idx
        for t in range(start_idx, len(timestamps)):
            if timestamps[t] >= stop_time:
                stop_idx = t
                break

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

        # Use MAX instead of MEAN
        neural_max = []
        for neuron_idx in range(neural_chunk.shape[1]):
            neuron_data = neural_chunk[:, neuron_idx]
            maximum = np.max(neuron_data)  # Changed from mean to max
            neural_max.append(maximum)

        all_trials.append(neural_max)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)

    print("Data shape:", X.shape)
    print("Number of trials:", len(y))

    return X, y, condition_names

# Test with mean (regular way) - using exact clip duration
print("\nUsing MEAN activity (our usual method)")
neuron_list = []
for i in range(100):
    neuron_list.append(i)

X_mean, y_mean, cond_map = get_data_for_decoding(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list=neuron_list,
    target_conditions=target_conditions,
    window_size=None
)
acc_mean, _, _, _ = train_decoder(X_mean, y_mean, cond_map)

# Test with max - using exact clip duration
print("\nUsing MAX activity (peak response)")
X_max, y_max, cond_map = get_data_with_max(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list=neuron_list,
    target_conditions=target_conditions,
    window_size=None
)
acc_max, _, _, _ = train_decoder(X_max, y_max, cond_map)


# ============================================================================
# TEST 5
# ============================================================================
print("\n" + "="*60)
print("TEST 5: Fixed 5s Window vs Exact Clip Duration")
print("="*60)

# Use 100 neurons
neuron_list = []
for i in range(100):
    neuron_list.append(i)

# Test with fixed 5s window
print("\nUsing FIXED 5-second window")

# Make a mapping from condition name to number
condition_names = {}
label_number = 0
for cond in target_conditions:
    condition_names[cond] = label_number
    label_number = label_number + 1

# Store data here
all_trials_fixed = []
all_labels_fixed = []

# Go through each stimulus presentation
for i in range(len(clip_starts)):
    stim_type = clip_types[i]

    # Skip if not in our target conditions
    if stim_type not in target_conditions:
        continue

    # Find the time window - FIXED 5 seconds
    start_time = clip_starts[i]
    stop_time = start_time + 5.0

    # Convert times to indices
    start_idx = 0
    for t in range(len(timestamps)):
        if timestamps[t] >= start_time:
            start_idx = t
            break

    stop_idx = start_idx
    for t in range(start_idx, len(timestamps)):
        if timestamps[t] >= stop_time:
            stop_idx = t
            break

    # Get neural data for this window
    neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

    # Average across time to get one number per neuron
    neural_average = []
    for neuron_idx in range(neural_chunk.shape[1]):
        neuron_data = neural_chunk[:, neuron_idx]
        avg = np.mean(neuron_data)
        neural_average.append(avg)

    # Store this trial
    all_trials_fixed.append(neural_average)
    all_labels_fixed.append(condition_names[stim_type])

# Convert to arrays
X_fixed = np.array(all_trials_fixed)
y_fixed = np.array(all_labels_fixed)

print("Data shape:", X_fixed.shape)
print("Number of trials:", len(y_fixed))

# Train decoder
acc_fixed, _, _, _ = train_decoder(X_fixed, y_fixed, condition_names)


# Test with exact clip duration
print("\nUsing EXACT clip duration")

# Store data here
all_trials_exact = []
all_labels_exact = []

# Go through each stimulus presentation
for i in range(len(clip_starts)):
    stim_type = clip_types[i]

    # Skip if not in our target conditions
    if stim_type not in target_conditions:
        continue

    # Find the time window - EXACT DURATION from clip_stops
    start_time = clip_starts[i]
    stop_time = clip_stops[i]  # Use the actual stop time

    # Convert times to indices
    start_idx = 0
    for t in range(len(timestamps)):
        if timestamps[t] >= start_time:
            start_idx = t
            break

    stop_idx = start_idx
    for t in range(start_idx, len(timestamps)):
        if timestamps[t] >= stop_time:
            stop_idx = t
            break

    # Get neural data for this window
    neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])

    # Average across time to get one number per neuron
    neural_average = []
    for neuron_idx in range(neural_chunk.shape[1]):
        neuron_data = neural_chunk[:, neuron_idx]
        avg = np.mean(neuron_data)
        neural_average.append(avg)

    # Store this trial
    all_trials_exact.append(neural_average)
    all_labels_exact.append(condition_names[stim_type])

# Convert to arrays
X_exact = np.array(all_trials_exact)
y_exact = np.array(all_labels_exact)

print("Data shape:", X_exact.shape)
print("Number of trials:", len(y_exact))

# Train decoder
acc_exact, _, _, _ = train_decoder(X_exact, y_exact, condition_names)



# ============================================================================
# TEST 6
# ============================================================================
print("\n" + "="*60)
print("TEST 6: Nueron 4 fixed time duration vs exact")
print("="*60)

# Test neuron 4 with fixed 5s window
print("\nNeuron 4 with FIXED 5-second window")
X_n4_fixed, y_n4_fixed, cond_map = get_data_for_decoding(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list=[3],  # Neuron 4
    target_conditions=target_conditions,
    window_size=5.0
)
acc_n4_fixed, _, _, _ = train_decoder(X_n4_fixed, y_n4_fixed, cond_map)

# Test neuron 4 with exact clip duration
print("\nNeuron 4 with EXACT clip duration")

# Store data here
all_trials_n4_exact = []
all_labels_n4_exact = []

# Make condition mapping
condition_names = {}
label_number = 0
for cond in target_conditions:
    condition_names[cond] = label_number
    label_number = label_number + 1

# Go through each stimulus presentation
for i in range(len(clip_starts)):
    stim_type = clip_types[i]

    # Skip if not in our target conditions
    if stim_type not in target_conditions:
        continue

    # Find the time window - EXACT DURATION from clip_stops
    start_time = clip_starts[i]
    stop_time = clip_stops[i]  # Use the actual stop time

    # Convert times to indices
    start_idx = 0
    for t in range(len(timestamps)):
        if timestamps[t] >= start_time:
            start_idx = t
            break

    stop_idx = start_idx
    for t in range(start_idx, len(timestamps)):
        if timestamps[t] >= stop_time:
            stop_idx = t
            break

    # Get neural data for this window - ONLY NEURON 4
    neural_chunk = np.array(rs.data[start_idx:stop_idx, 3])  # Index 3 = neuron 4

    # Average across time
    avg = np.mean(neural_chunk)

    # Store this trial
    all_trials_n4_exact.append([avg])  # Wrap in list to make it 2D
    all_labels_n4_exact.append(condition_names[stim_type])

# Convert to arrays
X_n4_exact = np.array(all_trials_n4_exact)
y_n4_exact = np.array(all_labels_n4_exact)

print("Data shape:", X_n4_exact.shape)
print("Number of trials:", len(y_n4_exact))

# Train decoder
acc_n4_exact, _, _, _ = train_decoder(X_n4_exact, y_n4_exact, condition_names)


Connected to dataset
Total neurons: 1455
Total timepoints: 40000
Total stimulus presentations: 384
   Cinematic : 128 presentations
   Rendered : 128 presentations
   sports1m : 128 presentations

TEST 1: How many neurons do we need

Testing with 10 neurons
Data shape: (384, 10)
Number of trials: 384
Training trials: 307
Test trials: 77
Test accuracy: 0.5454545454545454
Result: 54.5 %

Testing with 50 neurons
Data shape: (384, 50)
Number of trials: 384
Training trials: 307
Test trials: 77
Test accuracy: 0.5714285714285714
Result: 57.1 %

Testing with 100 neurons
Data shape: (384, 100)
Number of trials: 384
Training trials: 307
Test trials: 77
Test accuracy: 0.6363636363636364
Result: 63.6 %

Testing with 200 neurons
Data shape: (384, 200)
Number of trials: 384
Training trials: 307
Test trials: 77
Test accuracy: 0.5714285714285714
Result: 57.1 %

Testing with 500 neurons
Data shape: (384, 500)
Number of trials: 384
Training trials: 307
Test trials: 77
Test accuracy: 0.4675324675324675
R